In [ ]:
import torch
from evo2 import Evo2

model = Evo2('evo2_7b')
DEVICE = 'cuda:0'
LAYER = 'blocks.28.mlp.l3'  # 


def embed_gene(
    seq: str,
    layer: str = LAYER,
    pool: str = 'mean',           # 'mean' | 'last' | 'none'
    both_strands: bool = True,
) -> torch.Tensor:
    """
    Embed a single gene sequence with Evo 2.

    seq: the gene sequence as a string (ACGT). Ideally include some flanking
         context (promoter + RBS upstream, terminator downstream) — Evo 2
         was trained on genomic context, not isolated CDSs.
    Returns: [d_model] if pooled, else [L, d_model].
    """
    def _forward(s: str) -> torch.Tensor:
        ids = torch.tensor(
            model.tokenizer.tokenize(s), dtype=torch.int
        ).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            _, emb = model(ids, return_embeddings=True, layer_names=[layer])
        return emb[layer].squeeze(0).float().cpu()  # [L, d_model]

    h = _forward(seq)

    if both_strands:
        rc = seq.translate(str.maketrans('ACGTacgt', 'TGCATGCA'))[::-1]
        h_rc = _forward(rc)
        # average token-level reps after flipping rc back to forward orientation
        h = (h + h_rc.flip(0)) / 2

    if pool == 'mean':
        return h.mean(dim=0)            # [d_model]
    elif pool == 'last':
        return h[-1]                    # [d_model]
    elif pool == 'none':
        return h                        # [L, d_model]
    else:
        raise ValueError(pool)


# usage
gene_seq = "ATG...TAA"  # ideally with ~500–1000 bp flanking on each side
vec = embed_gene(gene_seq)
print(vec.shape)  # torch.Size([d_model])intermediate layer — recommended over final
